# Reprodução limpa da CV e ablação — TP1 E2

Executar a única célula de código em uma sessão nova do Google Colab após integrar a atualização na main. A rotina usa apenas o repositório e a versão 2 da base pública; não depende de upload de ZIPs, células antigas, variáveis previamente definidas ou diretórios da máquina local.

Mantém os IDs congelados 2.800/600/600 e a semente 42. Refaz auditoria, textura, HOG, intensidade, CV, ablação e figuras. Os modelos são avaliados somente nas três dobras do treino; validação e teste externos não são usados para seleção.

O protocolo é provisório por imagem, sem independência por paciente comprovada. A flag é uma declaração da limitação, não uma autorização do professor. Havendo chave verificável, utilizar a opção --groups-csv do script e revisar os splits.

O download tem aproximadamente 9,3 GiB; a CV em CPU pode demorar horas. A saída informa início e conclusão por modelo/dobra. As porcentagens gerais correspondem a etapas e não estimam tempo restante. Uma desconexão pode interromper a sessão; salvar o ZIP ao concluir.

Saída: verificacao_limpa_TP1.zip, com auditoria, matrizes, metadados, OOF, métricas, versões, figuras e relatório de comparação. A execução não encerra afiliação, avaliação final externa, esclarecimento do protocolo nem assinaturas.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import tempfile
from google.colab import files

print('0% — Criando cópia limpa do repositório.')
work_repro = Path(tempfile.mkdtemp(prefix='tp1_repro_', dir='/content'))
repo_repro = work_repro / 'repo'
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main',
                'https://github.com/RaulFH0/tp1-e2-bone-age.git', str(repo_repro)], check=True)
if not (repo_repro / 'src/10_reproduce_from_raw.py').is_file():
    raise RuntimeError('A main ainda não contém a atualização de reprodução. Integrar o PR antes de executar este notebook.')
commit_repro = subprocess.check_output(['git', '-C', str(repo_repro), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(repo_repro / 'requirements-colab.txt')], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
               cwd=repo_repro, check=True)
print('10% — Dependências instaladas e testes aprovados. Commit:', commit_repro)
print('Iniciando a reprodução a partir das imagens. Download, extração e treinamentos têm progresso próprio.')
subprocess.run([sys.executable, '-u', 'src/10_reproduce_from_raw.py', '--allow-image-folds'],
               cwd=repo_repro, check=True)
resultado_repro = repo_repro / 'results/reproducibilidade/verificacao_reproducibilidade.json'
relatorio_repro = json.loads(resultado_repro.read_text())
print('Diferença máxima das métricas:', relatorio_repro['max_absolute_metric_difference'])
print('Protocolo:', relatorio_repro['protocol'])
zip_repro = repo_repro / 'results/verificacao_limpa_TP1.zip'
print('100% desta execução — pacote pronto. Salvar também uma cópia do notebook executado.')
files.download(str(zip_repro))
